# Final Assignment: Extracting and Visualizing Stock Data

**Role:** Data scientist at an investment firm.
**Goal:** Extract historical share prices (with `yfinance`) and quarterly revenue (with web scraping) for **Tesla (TSLA)** and **GameStop (GME)**, then build dashboards that compare stock price against revenue.

| Question | Task |
|---|---|
| 1 | Extract Tesla stock data using `yfinance` |
| 2 | Extract Tesla revenue data using web scraping |
| 3 | Extract GameStop stock data using `yfinance` |
| 4 | Extract GameStop revenue data using web scraping |
| 5 | Tesla stock and revenue dashboard |
| 6 | GameStop stock and revenue dashboard |
| 7 | Share this notebook on GitHub |

## Setup: install and import libraries

In [1]:
# Run once if the libraries are missing (Skills Network Labs / local Jupyter)
!pip install yfinance beautifulsoup4 lxml html5lib requests pandas plotly nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 1.0 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 601.0 kB/s  0:00:16 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 536.7 kB/s  0:00:05 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [yfinance]7/9 [curl_cffi]


In [2]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.width", 120)

## Helper: `make_graph`

Plots historical share price (top) and historical revenue (bottom) on two stacked charts.
Data is limited to the same cut-off dates used in the course lab (share price up to 2021-06-14, revenue up to 2021-04-30).

In [3]:
def make_graph(stock_data, revenue_data, stock):
    """Draw share price and revenue dashboards for one company."""
    stock_df = stock_data.copy()
    rev_df = revenue_data.copy()

    # Normalise dates (yfinance returns timezone-aware timestamps)
    stock_df["Date"] = pd.to_datetime(stock_df["Date"], utc=True).dt.tz_localize(None)
    rev_df["Date"] = pd.to_datetime(rev_df["Date"])

    stock_specific = stock_df[stock_df["Date"] <= pd.Timestamp("2021-06-14")]
    revenue_specific = rev_df[rev_df["Date"] <= pd.Timestamp("2021-04-30")]

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=("Historical Share Price", "Historical Revenue"),
        vertical_spacing=0.3,
    )
    fig.add_trace(
        go.Scatter(x=stock_specific["Date"], y=stock_specific["Close"].astype("float"), name="Share Price"),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=revenue_specific["Date"], y=revenue_specific["Revenue"].astype("float"), name="Revenue"),
        row=2, col=1,
    )
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

## Helper: scrape a quarterly revenue table

Both revenue pages contain a table of `Date` / `Revenue` rows. This helper parses the HTML with `BeautifulSoup`,
cleans the `$` and `,` characters out of the revenue values, and drops empty rows.

In [4]:
def scrape_revenue(url):
    """Download a page and return a DataFrame with columns Date and Revenue."""
    html_data = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html_data, "html.parser")

    # Pick the table whose text mentions "Quarterly Revenue"; fall back to the second table on the page
    tables = soup.find_all("table")
    table = next((t for t in tables if "Quarterly Revenue" in t.get_text()), tables[1])

    rows = []
    for tr in table.find_all("tr"):
        cells = tr.find_all("td")
        if len(cells) >= 2:
            rows.append({"Date": cells[0].get_text(strip=True), "Revenue": cells[1].get_text(strip=True)})

    revenue = pd.DataFrame(rows, columns=["Date", "Revenue"])
    revenue["Revenue"] = revenue["Revenue"].str.replace(r"[\$,]", "", regex=True)
    revenue = revenue[revenue["Revenue"] != ""].reset_index(drop=True)   # drop empty strings
    return revenue

---
## Question 1: Use yfinance to Extract Stock Data

Create a ticker object for Tesla (`TSLA`), pull the maximum available history, reset the index and display the first five rows.

In [5]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


---
## Question 2: Use Webscraping to Extract Tesla Revenue Data

Download the Tesla revenue page, extract the table into `tesla_revenue` (columns `Date`, `Revenue`), clean it and display the last five rows.

In [6]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = scrape_revenue(url)
tesla_revenue.tail()

,Date,Revenue
48,2010-09-30,31
49,2010-06-30,28
50,2010-03-31,21
51,2009-09-30,46
52,2009-06-30,27


---
## Question 3: Use yfinance to Extract Stock Data

Create a ticker object for GameStop (`GME`), pull the maximum available history, reset the index and display the first five rows.

In [7]:
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620129,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


---
## Question 4: Use Webscraping to Extract GME Revenue Data

Download the GameStop revenue page, extract the table into `gme_revenue` (columns `Date`, `Revenue`), clean it and display the last five rows.

In [8]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
gme_revenue = scrape_revenue(url)
gme_revenue.tail()

,Date,Revenue
57,2006-01-31,1667
58,2005-10-31,534
59,2005-07-31,416
60,2005-04-30,475
61,2005-01-31,709


---
## Question 5: Plot Tesla Stock Graph

Use `make_graph` to plot the Tesla share price and revenue, with the title `Tesla`.

In [9]:
make_graph(tesla_data, tesla_revenue, "Tesla")

---
## Question 6: Plot GameStop Stock Graph

Use `make_graph` to plot the GameStop share price and revenue, with the title `GameStop`.

In [10]:
make_graph(gme_data, gme_revenue, "GameStop")

---
## Question 7: Sharing your Assignment Notebook

This notebook is published on GitHub:
https://github.com/ayyapparaja227-arch/PythonProject_Coursera